# Numerical Methods for Stochastic Processes

This notebook is the project walkthrough. It groups the original exercises by
mathematical topic and adds reproducible experiments and interpretation.

**Reproducibility:** all stochastic experiments below use explicit seeds.


In [ ]:
from pathlib import Path
import sys
ROOT = Path.cwd() if (Path.cwd() / 'src').exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT / 'src'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from stochastic_numerics import (
    box_muller, gaussian_accept_reject, marsaglia_polar, cauchy,
    brownian_motion, ito_integral_moments, strong_convergence_experiment,
)

SEED = 2026
rng = np.random.default_rng(SEED)
plt.style.use('seaborn-v0_8-whitegrid')


## 1. Random variate generation

The first exercises compare transformations and rejection sampling for
standard normal random variables, alongside inverse-transform sampling for a
Cauchy distribution. The heavy tails of the Cauchy distribution mean its
population mean and variance do not exist; sample means and variances are
therefore unstable summaries.


In [ ]:
n = 100_000
normal_samples = {
    "Box–Muller": box_muller(n, np.random.default_rng(SEED)),
    "Marsaglia polar": marsaglia_polar(n, np.random.default_rng(SEED + 1)),
}
rejection_samples, acceptance_rate = gaussian_accept_reject(
    n, np.random.default_rng(SEED + 2)
)
normal_samples["Acceptance–rejection"] = rejection_samples

summary = pd.DataFrame({
    name: {"sample mean": x.mean(), "sample variance": x.var(),
           "sample skewness": pd.Series(x).skew()}
    for name, x in normal_samples.items()
}).T
display(summary)
print(f"Empirical acceptance rate: {acceptance_rate:.3f}")


In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
for name, x in normal_samples.items():
    ax.hist(x, bins=100, density=True, histtype="step", label=name)
grid = np.linspace(-4, 4, 400)
ax.plot(grid, np.exp(-grid**2 / 2) / np.sqrt(2*np.pi), "k--", label="N(0,1)")
ax.set(title="Standard normal sampling methods", xlabel="x", ylabel="Density")
ax.legend()
plt.show()


## 2. Brownian motion

A standard Brownian motion is built by cumulatively summing independent
increments distributed as `N(0, dt)`.


In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
time = np.linspace(0, 1, 1001)
for k in range(5):
    path = brownian_motion(1000, rng=np.random.default_rng(SEED + k))
    ax.plot(time, path, linewidth=1, label=f"Path {k+1}")
ax.set(title="Sample paths of Brownian motion", xlabel="Time", ylabel="W(t)")
ax.legend(ncol=2)
plt.show()


## 3. Monte Carlo approximation of an Itô integral

Consider `I = integral_0^1 t W_t dW_t`. The left-point sum is an Itô
discretisation. Itô isometry gives the reference values `E[I] = 0` and
`E[I²] = 1/4`. Batching limits peak memory use.


In [ ]:
rows = []
for n_steps in (16, 32, 64, 128, 256):
    mean_i, mean_i2 = ito_integral_moments(
        n_steps=n_steps, n_paths=100_000, batch_size=5_000,
        seed=SEED + n_steps
    )
    rows.append((n_steps, mean_i, mean_i2, abs(mean_i2 - 0.25)))
ito_results = pd.DataFrame(
    rows, columns=["time steps", "estimated E[I]", "estimated E[I²]", "abs. error in E[I²]"]
)
display(ito_results)


## 4. Euler–Maruyama and strong convergence

For `dX_t = a(X_t)dt + b(X_t)dW_t`, Euler–Maruyama uses

`X[n+1] = X[n] + a(X[n]) dt + b(X[n]) ΔW[n]`.

The experiment below reproduces the exercise coefficients
`a(x) = log(1 + x²)` and `b(x) = x 1{x > 0}`. Adjacent resolutions share
the same underlying Brownian path (coarse increments are sums of fine
increments), which is essential for a strong-error comparison.


In [ ]:
drift = lambda x: np.log1p(x**2)
diffusion = lambda x: x * (x > 0)

steps, errors, pairwise_rates = strong_convergence_experiment(
    drift, diffusion, x0=1.0, horizon=1.0,
    base_steps=10, refinements=5, n_paths=30_000,
    batch_size=2_000, seed=SEED
)
convergence = pd.DataFrame({
    "steps": steps[:-1], "dt": 1.0 / steps[:-1],
    "coupled L2 difference": errors,
})
display(convergence)
print("Pairwise empirical log-log slopes:", pairwise_rates)


In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
ax.loglog(convergence["dt"], convergence["coupled L2 difference"],
          marker="o", label="Adjacent-level L2 difference")
ax.set(title="Euler–Maruyama refinement experiment",
       xlabel="Time step size", ylabel="L2 difference")
ax.invert_xaxis()
ax.legend()
plt.show()


## 5. Interpretation and limitations

Use this section to explain what the numerical results show, rather than
only reporting output. In particular:

- Are the normal generators consistent with a standard normal distribution?
- How does the Cauchy sample differ from a Gaussian sample?
- Does the estimated second moment of the Itô sum approach 1/4 as the grid is
  refined? Separate discretisation effects from Monte Carlo sampling error.
- What empirical convergence slope do you observe for the chosen SDE?
- How do sample size, time-step range and batching affect runtime and error?

The strong-convergence slopes are empirical estimates, not universal
guarantees. State the theoretical assumptions separately from what this
finite-sample experiment demonstrates.
